In [ ]:
# ================================================================
# DEEPSEEK-CODER MALWARE CLASSIFICATION
# RESTORE SAVED ADAPTER + FULL VALIDATION EVALUATION
# ================================================================

import os
import glob
import re
import json
import shutil
import warnings
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

EXPECTED_VALIDATION_SIZE = 5898

BATCH_SIZE = 4
MAX_NEW_TOKENS = 20

ADAPTER_DIR = "/kaggle/working/deepseek_final_adapter"

OUTPUT_DIR = "/kaggle/working/deepseek_validation_results"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = [
    "Benign",
    "RedLineStealer",
    "Downloader",
    "RAT",
    "BankingTrojan",
    "SnakeKeyLogger",
    "Spyware"
]

LABEL_MAP = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

ID_TO_LABEL = LABEL_MAP

print("=" * 70)
print("DEEPSEEK MALWARE VALIDATION")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Enable GPU in Kaggle settings.")

print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# 2. LOCATE DATASET
# ================================================================

csv_files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

valid_csv_files = []

for path in csv_files:

    try:
        columns = pd.read_csv(path, nrows=0).columns

        if {"SHA256", "Type"}.issubset(columns):
            valid_csv_files.append(path)

    except Exception:
        continue

if not valid_csv_files:
    raise FileNotFoundError(
        "Malware_Dataset.csv was not found in Kaggle input."
    )

CSV_PATH = valid_csv_files[0]

print("\nDataset:", CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

assert df["SHA256"].is_unique
assert df.isnull().sum().sum() == 0

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

assert len(FEATURE_COLUMNS) == 318

assert all(
    pd.api.types.is_numeric_dtype(df[col])
    for col in FEATURE_COLUMNS
)

df["Type"] = pd.to_numeric(
    df["Type"],
    errors="raise"
).astype(int)

assert set(df["Type"].unique()) == set(LABEL_MAP.keys())

print("Dataset validation successful.")
print("Samples:", len(df))
print("PE features:", len(FEATURE_COLUMNS))


# ================================================================
# 3. RECONSTRUCT ORIGINAL STRATIFIED SPLIT
# ================================================================

development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["Type"]
)

train_df, validation_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=SEED,
    stratify=development_df["Type"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

assert len(train_df) == 17693
assert len(validation_df) == EXPECTED_VALIDATION_SIZE
assert len(test_df) == 5898

assert set(train_df["SHA256"]).isdisjoint(
    validation_df["SHA256"]
)

assert set(train_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

assert set(validation_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

print("\nOriginal split reconstructed successfully.")

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Testing:", len(test_df))

print("\nValidation class distribution:")
print(validation_df["Type"].value_counts().sort_index())


# ================================================================
# 4. LOCATE AND RESTORE SAVED ADAPTER FILES
# ================================================================

all_files = glob.glob(
    "/kaggle/input/**/*",
    recursive=True
)

def find_file(prefix, extension):

    matches = [
        path for path in all_files
        if os.path.isfile(path)
        and os.path.basename(path).startswith(prefix)
        and path.endswith(extension)
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find {prefix}*{extension} in Kaggle input."
        )

    if len(matches) > 1:
        print(f"Multiple matches for {prefix}; using:", matches[0])

    return matches[0]


adapter_config_source = find_file(
    "adapter_config",
    ".json"
)

adapter_weights_source = find_file(
    "adapter_model",
    ".safetensors"
)

tokenizer_source = find_file(
    "tokenizer",
    ".json"
)

tokenizer_config_source = find_file(
    "tokenizer_config",
    ".json"
)

files_to_restore = {
    adapter_config_source: "adapter_config.json",
    adapter_weights_source: "adapter_model.safetensors",
    tokenizer_source: "tokenizer.json",
    tokenizer_config_source: "tokenizer_config.json"
}

print("\nRestoring adapter files...")

for source, destination_name in files_to_restore.items():

    destination = os.path.join(
        ADAPTER_DIR,
        destination_name
    )

    shutil.copy2(source, destination)

    print(
        destination_name,
        "|",
        round(os.path.getsize(destination) / (1024 ** 2), 2),
        "MB"
    )

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json"
]

for filename in required_files:

    assert os.path.exists(
        os.path.join(ADAPTER_DIR, filename)
    ), f"Missing adapter file: {filename}"

print("\nAdapter files restored successfully.")


# ================================================================
# 5. LOAD SAVED TOKENIZER
# ================================================================

print("\nLoading saved tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Tokenizer loaded.")
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)


# ================================================================
# 6. RECREATE ORIGINAL FEATURE GROUPS
# ================================================================

API_FEATURES = FEATURE_COLUMNS[:200]

DLL_FEATURES = FEATURE_COLUMNS[200:227]

HEADER_FEATURES = FEATURE_COLUMNS[227:273]

SECTION_FEATURES = FEATURE_COLUMNS[273:318]

assert len(API_FEATURES) == 200
assert len(DLL_FEATURES) == 27
assert len(HEADER_FEATURES) == 46
assert len(SECTION_FEATURES) == 45


# ================================================================
# 7. ORIGINAL TRAINING PROMPT TEMPLATE
# ================================================================

SYSTEM_PROMPT = """You are a Windows PE malware classification assistant.

Classify the executable using only its provided static PE features.

Instructions:
1. Analyze all provided feature values.
2. Zero values are meaningful and must not be ignored.
3. Do not invent missing information.
4. Do not infer unsupported runtime behavior.
5. Select exactly one malware family.

Available classes:
Benign, RedLineStealer, Downloader, RAT,
BankingTrojan, SnakeKeyLogger, Spyware.
"""


def format_features(row, columns):

    return "\n".join(
        f"{col}={row[col]}"
        for col in columns
    )


def create_prompt(row):

    return f"""### SYSTEM:
{SYSTEM_PROMPT}

### USER:
Analyze the following Windows PE static feature profile.

<API_FUNCTIONS>
{format_features(row, API_FEATURES)}
</API_FUNCTIONS>

<DLL_IMPORTS>
{format_features(row, DLL_FEATURES)}
</DLL_IMPORTS>

<PE_HEADER>
{format_features(row, HEADER_FEATURES)}
</PE_HEADER>

<PE_SECTIONS>
{format_features(row, SECTION_FEATURES)}
</PE_SECTIONS>

Classify this executable into exactly one of the seven malware families.

### ASSISTANT:
"""


# ================================================================
# 8. LOAD BASE MODEL IN 4-BIT NF4
# ================================================================

print("\nLoading DeepSeek-Coder base model...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print("Base model loaded.")

# Load the saved trained LoRA adapter.

print("\nLoading trained LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)

model.eval()

print("Trained adapter loaded successfully.")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


# ================================================================
# 9. PREPARE VALIDATION DATA
# ================================================================

validation_prompts = []

validation_labels = []

validation_hashes = []

for _, row in validation_df.iterrows():

    prompt = create_prompt(row)

    encoded = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False
    )

    input_ids = encoded["input_ids"]

    if len(input_ids) > 4096:
        raise ValueError(
            f"Validation prompt exceeds 4096 tokens: {len(input_ids)}"
        )

    validation_prompts.append(input_ids)

    validation_labels.append(
        LABEL_MAP[int(row["Type"])]
    )

    validation_hashes.append(
        str(row["SHA256"])
    )

assert len(validation_prompts) == EXPECTED_VALIDATION_SIZE

print("\nValidation prompts prepared.")
print("Total prompts:", len(validation_prompts))
print(
    "Maximum prompt length:",
    max(len(x) for x in validation_prompts)
)


# ================================================================
# 10. PREDICTION PARSER
# ================================================================

# Longest class names first to avoid partial matching.

SORTED_CLASSES = sorted(
    CLASS_NAMES,
    key=len,
    reverse=True
)

CLASS_PATTERN = re.compile(
    r"\b(" + "|".join(
        re.escape(name)
        for name in SORTED_CLASSES
    ) + r")\b",
    flags=re.IGNORECASE
)


def extract_prediction(text):

    match = CLASS_PATTERN.search(text)

    if match is None:
        return "INVALID"

    predicted = match.group(1)

    for class_name in CLASS_NAMES:

        if predicted.lower() == class_name.lower():
            return class_name

    return "INVALID"


# ================================================================
# 11. RUN FULL VALIDATION INFERENCE
# ================================================================

print("\nStarting validation inference...")

predictions = []

model_device = next(model.parameters()).device

total_batches = (
    len(validation_prompts) + BATCH_SIZE - 1
) // BATCH_SIZE

for batch_start in tqdm(
    range(0, len(validation_prompts), BATCH_SIZE),
    total=total_batches,
    desc="Validation"
):

    batch_prompts = validation_prompts[
        batch_start:batch_start + BATCH_SIZE
    ]

    batch_max_length = max(
        len(x) for x in batch_prompts
    )

    padded_ids = []
    attention_masks = []

    for ids in batch_prompts:

        padding_length = batch_max_length - len(ids)

        padded_ids.append(
            [tokenizer.pad_token_id] * padding_length + ids
        )

        attention_masks.append(
            [0] * padding_length + [1] * len(ids)
        )

    input_ids = torch.tensor(
        padded_ids,
        dtype=torch.long,
        device=model_device
    )

    attention_mask = torch.tensor(
        attention_masks,
        dtype=torch.long,
        device=model_device
    )

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    # Decode only newly generated tokens.

    generated_tokens = outputs[
        :, input_ids.shape[1]:
    ]

    decoded_outputs = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    for generated_text in decoded_outputs:

        prediction = extract_prediction(
            generated_text
        )

        predictions.append(prediction)

    del input_ids
    del attention_mask
    del outputs
    del generated_tokens

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Save intermediate progress every 100 batches.

    current_batch = (
        batch_start // BATCH_SIZE
    ) + 1

    if current_batch % 100 == 0:

        partial_df = pd.DataFrame({
            "SHA256": validation_hashes[:len(predictions)],
            "Actual": validation_labels[:len(predictions)],
            "Predicted": predictions
        })

        partial_df.to_csv(
            os.path.join(
                OUTPUT_DIR,
                "validation_predictions_partial.csv"
            ),
            index=False
        )

        print(
            f"\nProgress saved: {len(predictions)}/"
            f"{EXPECTED_VALIDATION_SIZE}"
        )


# ================================================================
# 12. VERIFY PREDICTION COUNT
# ================================================================

assert len(predictions) == EXPECTED_VALIDATION_SIZE, (
    f"Expected {EXPECTED_VALIDATION_SIZE} predictions, "
    f"received {len(predictions)}"
)

print("\nInference completed successfully.")

actual = validation_labels

invalid_count = predictions.count("INVALID")

print("Total evaluated:", len(predictions))
print("Invalid predictions:", invalid_count)


# ================================================================
# 13. CALCULATE EVALUATION METRICS
# ================================================================

accuracy = accuracy_score(
    actual,
    predictions
)

precision_macro = precision_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="macro",
    zero_division=0
)

f1_weighted = f1_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="weighted",
    zero_division=0
)

balanced_accuracy = balanced_accuracy_score(
    actual,
    predictions
)


# ================================================================
# 14. PRINT FINAL METRICS
# ================================================================

print("\n" + "=" * 70)
print("DEEPSEEK-CODER VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy:           {accuracy:.4f}")
print(f"Macro Precision:    {precision_macro:.4f}")
print(f"Macro Recall:       {recall_macro:.4f}")
print(f"Macro F1:           {f1_macro:.4f}")
print(f"Weighted F1:        {f1_weighted:.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy:.4f}")

print(f"Invalid Predictions: {invalid_count}")

print("\nClassification Report:\n")

print(
    classification_report(
        actual,
        predictions,
        labels=CLASS_NAMES,
        zero_division=0,
        digits=4
    )
)


# ================================================================
# 15. CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    actual,
    predictions,
    labels=CLASS_NAMES
)

confusion_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print("\nConfusion Matrix:\n")

print(confusion_df)


# ================================================================
# 16. SAVE VALIDATION RESULTS
# ================================================================

results_df = pd.DataFrame({
    "SHA256": validation_hashes,
    "Actual": actual,
    "Predicted": predictions
})

results_path = os.path.join(
    OUTPUT_DIR,
    "deepseek_validation_predictions.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

confusion_path = os.path.join(
    OUTPUT_DIR,
    "deepseek_validation_confusion_matrix.csv"
)

confusion_df.to_csv(confusion_path)

metrics = {
    "model": MODEL_NAME,
    "adapter": ADAPTER_DIR,
    "validation_samples": len(actual),
    "accuracy": float(accuracy),
    "macro_precision": float(precision_macro),
    "macro_recall": float(recall_macro),
    "macro_f1": float(f1_macro),
    "weighted_f1": float(f1_weighted),
    "balanced_accuracy": float(balanced_accuracy),
    "invalid_predictions": int(invalid_count)
}

metrics_path = os.path.join(
    OUTPUT_DIR,
    "deepseek_validation_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)


# ================================================================
# 17. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("VALIDATION EVALUATION COMPLETED")
print("=" * 70)

print("\nSaved files:")

print(results_path)
print(confusion_path)
print(metrics_path)

print("\nFinal result:")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Macro F1: {f1_macro * 100:.2f}%")
print(f"Weighted F1: {f1_weighted * 100:.2f}%")

print("\nThe test dataset has NOT been evaluated.")
print("The trained DeepSeek adapter has NOT been modified.")

print("\nAll done.")

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import pandas as pd
import numpy as np
import os

# Load saved validation predictions
PREDICTION_PATH = (
    "/kaggle/working/deepseek_validation_results/"
    "deepseek_validation_predictions.csv"
)

df = pd.read_csv(PREDICTION_PATH)

print("=" * 65)
print("DEEPSEEK VALIDATION ERROR ANALYSIS")
print("=" * 65)

print("\nDataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

# Display initial records
display(df.head())

# Identify prediction columns
print("\nPrediction column candidates:")
for col in df.columns:
    print(col, ":", df[col].dtype)


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/kaggle/working/deepseek_validation_results/"
    "deepseek_validation_predictions.csv"
)

# Identify incorrect predictions
errors = df[df["Actual"] != df["Predicted"]].copy()

print("=" * 65)
print("VALIDATION ERROR SUMMARY")
print("=" * 65)

print(f"\nTotal samples: {len(df)}")
print(f"Correct predictions: {len(df) - len(errors)}")
print(f"Incorrect predictions: {len(errors)}")
print(f"Error rate: {len(errors)/len(df)*100:.2f}%")

# Most frequent confusion patterns
print("\nTOP 15 CONFUSION PATTERNS")

confusions = (
    errors.groupby(["Actual", "Predicted"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

display(confusions.head(15))

# Class-wise error rates
print("\nCLASS-WISE ERROR ANALYSIS")

class_analysis = df.groupby("Actual").apply(
    lambda x: pd.Series({
        "Total": len(x),
        "Correct": (x["Actual"] == x["Predicted"]).sum(),
        "Incorrect": (x["Actual"] != x["Predicted"]).sum(),
        "Error_Rate_%": (x["Actual"] != x["Predicted"]).mean() * 100
    }),
    include_groups=False
)

display(class_analysis.round(2))

# Save detailed errors
errors.to_csv(
    "/kaggle/working/deepseek_validation_results/"
    "validation_misclassified_samples.csv",
    index=False
)

print("\nMisclassified samples saved successfully.")

In [ ]:
import pandas as pd

path = "/kaggle/working/deepseek_validation_results/deepseek_validation_predictions.csv"
df = pd.read_csv(path)

# Correctness indicator
df["Correct"] = df["Actual"] == df["Predicted"]

# Class-wise analysis without groupby.apply()
class_analysis = df.groupby("Actual").agg(
    Total=("Correct", "size"),
    Correct=("Correct", "sum")
)

class_analysis["Incorrect"] = (
    class_analysis["Total"] - class_analysis["Correct"]
)

class_analysis["Accuracy_%"] = (
    class_analysis["Correct"] / class_analysis["Total"] * 100
)

class_analysis["Error_Rate_%"] = (
    class_analysis["Incorrect"] / class_analysis["Total"] * 100
)

print("CLASS-WISE VALIDATION ERROR ANALYSIS")
display(class_analysis.round(2))

# Save results
class_analysis.to_csv(
    "/kaggle/working/deepseek_validation_results/"
    "validation_classwise_errors.csv"
)

print("Class-wise analysis saved successfully.")

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Feature groups
feature_groups = {
    "API Functions": df.columns[1:201].tolist(),
    "DLL Imports": df.columns[201:228].tolist(),
    "PE Header": df.columns[228:274].tolist(),
    "PE Sections": df.columns[274:319].tolist()
}

print("FEATURE GROUPS")
for group, features in feature_groups.items():
    print(f"{group}: {len(features)} features")

print("\nFeature-group analysis prepared successfully.")

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

feature_groups = {
    "API Functions": df.columns[1:201],
    "DLL Imports": df.columns[201:228],
    "PE Header": df.columns[228:274],
    "PE Sections": df.columns[274:319]
}

# Calculate non-zero feature percentages per class
results = []

for malware_type, group in df.groupby("Type"):
    row = {"Type": malware_type}

    for feature_group, columns in feature_groups.items():
        row[feature_group] = (
            group[columns].ne(0).to_numpy().mean() * 100
        )

    results.append(row)

distribution = pd.DataFrame(results)

print("NON-ZERO FEATURE DISTRIBUTION BY CLASS (%)")
display(distribution.round(2))

# Save results
distribution.to_csv(
    "/kaggle/working/deepseek_validation_results/"
    "feature_group_distribution.csv",
    index=False
)

print("Feature distribution analysis saved.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.preprocessing import LabelEncoder

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Exclude SHA256 and target
X = df.drop(columns=["SHA256", "Type"])
y = df["Type"]

print("Samples:", len(X))
print("Features:", X.shape[1])

# Calculate ANOVA F-scores
f_scores, p_values = f_classif(X, y)

# Calculate Mutual Information
mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42,
    n_neighbors=5
)

# Combine results
importance = pd.DataFrame({
    "Feature": X.columns,
    "ANOVA_F_Score": f_scores,
    "P_Value": p_values,
    "Mutual_Information": mi_scores
})

importance = importance.replace([np.inf, -np.inf], np.nan)
importance = importance.fillna(0)

# Rank features
importance["ANOVA_Rank"] = importance["ANOVA_F_Score"].rank(
    ascending=False,
    method="min"
)

importance["MI_Rank"] = importance["Mutual_Information"].rank(
    ascending=False,
    method="min"
)

importance = importance.sort_values(
    "Mutual_Information",
    ascending=False
)

print("\nTOP 25 FEATURES BY MUTUAL INFORMATION")
display(importance.head(25))

print("\nTOP 25 FEATURES BY ANOVA F-SCORE")
display(
    importance.sort_values(
        "ANOVA_F_Score",
        ascending=False
    ).head(25)
)

# Save results
output_path = (
    "/kaggle/working/deepseek_validation_results/"
    "individual_feature_importance.csv"
)

importance.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Focus on the three problematic classes
target_classes = [0, 5, 6]

class_names = {
    0: "Benign",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

# Exclude SHA256
feature_columns = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

results = []

for feature in feature_columns:

    row = {"Feature": feature}

    for class_id in target_classes:
        values = df.loc[df["Type"] == class_id, feature]

        row[class_names[class_id] + "_Mean"] = values.mean()
        row[class_names[class_id] + "_Median"] = values.median()
        row[class_names[class_id] + "_NonZero_%"] = (
            values.ne(0).mean() * 100
        )

    results.append(row)

comparison = pd.DataFrame(results)

# Calculate total separation across the three classes
comparison["Mean_Range"] = (
    comparison[
        ["Benign_Mean", "SnakeKeyLogger_Mean", "Spyware_Mean"]
    ].max(axis=1)
    -
    comparison[
        ["Benign_Mean", "SnakeKeyLogger_Mean", "Spyware_Mean"]
    ].min(axis=1)
)

# Rank features by separation
comparison = comparison.sort_values(
    "Mean_Range",
    ascending=False
)

print("TOP 30 FEATURES BY CLASS-WISE MEAN SEPARATION")
display(comparison.head(30).round(4))

# Save results
output_path = (
    "/kaggle/working/deepseek_validation_results/"
    "weak_class_feature_comparison.csv"
)

comparison.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif, mutual_info_classif

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Reconstruct original stratified split
train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    stratify=df["Type"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Type"],
    random_state=42
)

X_train = train_df.drop(columns=["SHA256", "Type"])
y_train = train_df["Type"]

classes = {
    0: "Benign",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

all_results = []

# One-vs-rest analysis
for class_id, class_name in classes.items():

    y_binary = (y_train == class_id).astype(int)

    f_scores, p_values = f_classif(X_train, y_binary)

    mi_scores = mutual_info_classif(
        X_train,
        y_binary,
        random_state=42,
        n_neighbors=5
    )

    result = pd.DataFrame({
        "Feature": X_train.columns,
        "Class": class_name,
        "ANOVA_F": f_scores,
        "P_Value": p_values,
        "Mutual_Information": mi_scores
    })

    all_results.append(result)

    print(f"\n{'='*65}")
    print(f"TOP 15 FEATURES: {class_name} vs REST")
    print("="*65)

    display(
        result.sort_values(
            "Mutual_Information",
            ascending=False
        ).head(15).round(6)
    )

# Combine and save
final_results = pd.concat(all_results, ignore_index=True)

output_path = (
    "/kaggle/working/deepseek_validation_results/"
    "weak_class_statistical_importance.csv"
)

final_results.to_csv(output_path, index=False)

print("\nTraining samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Saved:", output_path)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif, mutual_info_classif

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Reconstruct the original stratified split
train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    stratify=df["Type"],
    random_state=42
)

X_train = train_df.drop(columns=["SHA256", "Type"])
y_binary = (train_df["Type"] == 5).astype(int)

# Calculate feature importance
f_scores, p_values = f_classif(X_train, y_binary)

mi_scores = mutual_info_classif(
    X_train,
    y_binary,
    random_state=42,
    n_neighbors=5
)

results = pd.DataFrame({
    "Feature": X_train.columns,
    "ANOVA_F": f_scores,
    "P_Value": p_values,
    "Mutual_Information": mi_scores
})

print("TOP 15 FEATURES — SNAKEKEYLOGGER VS REST")

display(
    results.sort_values(
        "Mutual_Information",
        ascending=False
    ).head(15).round(6)
)

# Save results
output_path = (
    "/kaggle/working/deepseek_validation_results/"
    "snakekeylogger_feature_importance.csv"
)

results.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif, mutual_info_classif

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Reconstruct original training partition
train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    stratify=df["Type"],
    random_state=42
)

# Keep only SnakeKeyLogger (5) and Spyware (6)
pair_df = train_df[train_df["Type"].isin([5, 6])].copy()

X = pair_df.drop(columns=["SHA256", "Type"])
y = (pair_df["Type"] == 5).astype(int)

# Calculate statistical scores
f_scores, p_values = f_classif(X, y)

mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42,
    n_neighbors=5
)

results = pd.DataFrame({
    "Feature": X.columns,
    "ANOVA_F": f_scores,
    "P_Value": p_values,
    "Mutual_Information": mi_scores
})

results = results.sort_values(
    "Mutual_Information",
    ascending=False
)

print("SNAKEKEYLOGGER vs SPYWARE")
print("SnakeKeyLogger samples:", int(y.sum()))
print("Spyware samples:", int((y == 0).sum()))

display(results.head(25).round(6))

output_path = (
    "/kaggle/working/deepseek_validation_results/"
    "snakekeylogger_vs_spyware_features.csv"
)

results.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    stratify=df["Type"],
    random_state=42
)

# Focus on the two confused classes
pair_df = train_df[train_df["Type"].isin([5, 6])].copy()

pair_df["Class"] = pair_df["Type"].map({
    5: "SnakeKeyLogger",
    6: "Spyware"
})

features = [
    "TimeDateStamp",
    "rsrc_Misc_VirtualSize",
    "text_Misc_VirtualSize",
    "AddressOfEntryPoint",
    "SizeOfCode"
]

for feature in features:

    plt.figure(figsize=(9, 4))

    for class_name in ["SnakeKeyLogger", "Spyware"]:

        values = pair_df.loc[
            pair_df["Class"] == class_name,
            feature
        ]

        # Clip extreme values for readable visualization
        lower = values.quantile(0.01)
        upper = values.quantile(0.99)

        values = values.clip(lower, upper)

        plt.hist(
            values,
            bins=50,
            alpha=0.5,
            density=True,
            label=class_name
        )

    plt.title(f"Feature Distribution: {feature}")
    plt.xlabel(feature)
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()
    

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Reconstruct the original split
train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    stratify=df["Type"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Type"],
    random_state=42
)

# Keep only SnakeKeyLogger and Spyware
train_pair = train_df[train_df["Type"].isin([5, 6])]
val_pair = val_df[val_df["Type"].isin([5, 6])]

X_train = train_pair.drop(columns=["SHA256", "Type"])
y_train = train_pair["Type"]

X_val = val_pair.drop(columns=["SHA256", "Type"])
y_val = val_pair["Type"]

# Train classifier
model = ExtraTreesClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training pairwise classifier...")
model.fit(X_train, y_train)

# Validation predictions
predictions = model.predict(X_val)

print("\nPAIRWISE VALIDATION RESULTS")
print("Accuracy:", round(accuracy_score(y_val, predictions), 4))
print("Balanced Accuracy:", round(
    balanced_accuracy_score(y_val, predictions), 4
))

print("\nClassification Report:")
print(classification_report(
    y_val,
    predictions,
    labels=[5, 6],
    target_names=["SnakeKeyLogger", "Spyware"],
    digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, predictions, labels=[5, 6]))

print("\nValidation samples:", len(val_pair))

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

# Reconstruct original stratified split
train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    stratify=df["Type"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Type"],
    random_state=42
)

X_train = train_df.drop(columns=["SHA256", "Type"])
y_train = train_df["Type"]

X_val = val_df.drop(columns=["SHA256", "Type"])
y_val = val_df["Type"]

# Train seven-class Extra Trees
model = ExtraTreesClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training seven-class Extra Trees...")
model.fit(X_train, y_train)

predictions = model.predict(X_val)

# Metrics
print("\nFULL SEVEN-CLASS VALIDATION RESULTS")

print("Accuracy:",
      round(accuracy_score(y_val, predictions), 4))

print("Balanced Accuracy:",
      round(balanced_accuracy_score(y_val, predictions), 4))

print("Macro Precision:",
      round(precision_score(y_val, predictions, average="macro"), 4))

print("Macro Recall:",
      round(recall_score(y_val, predictions, average="macro"), 4))

print("Macro F1:",
      round(f1_score(y_val, predictions, average="macro"), 4))

print("\nCLASSIFICATION REPORT")

print(classification_report(
    y_val,
    predictions,
    digits=4
))

print("\nCONFUSION MATRIX")

print(confusion_matrix(
    y_val,
    predictions,
    labels=[0, 1, 2, 3, 4, 5, 6]
))

In [ ]:
import pandas as pd
import numpy as np

# Load DeepSeek validation predictions
deepseek = pd.read_csv(
    "/kaggle/working/deepseek_validation_results/deepseek_validation_predictions.csv"
)

# Save current Extra Trees predictions with sample identifiers
extra_trees = pd.DataFrame({
    "SHA256": val_df["SHA256"].values,
    "Actual": val_df["Type"].values,
    "Predicted": predictions
})

# Compare validation sample identities
deepseek_ids = set(deepseek["SHA256"])
extra_ids = set(extra_trees["SHA256"])

print("DeepSeek validation samples:", len(deepseek_ids))
print("Extra Trees validation samples:", len(extra_ids))
print("Common samples:", len(deepseek_ids & extra_ids))
print("Identical validation sets:", deepseek_ids == extra_ids)

# Save Extra Trees predictions
extra_trees.to_csv(
    "/kaggle/working/deepseek_validation_results/extra_trees_validation_predictions.csv",
    index=False
)

# Compare actual labels for common samples
comparison = deepseek.merge(
    extra_trees,
    on="SHA256",
    suffixes=("_DeepSeek", "_ExtraTrees")
)

print("\nActual label agreement:",
      (comparison["Actual_DeepSeek"] ==
       comparison["Actual_ExtraTrees"]).all())

print("Saved Extra Trees predictions.")

In [ ]:
import os

base_path = "/kaggle/working"

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
import pandas as pd

dataset_path = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"
pred_path = "/kaggle/working/deepseek_validation_results/deepseek_validation_predictions.csv"

df = pd.read_csv(dataset_path)
deepseek_preds = pd.read_csv(pred_path)

# Match dataset rows to DeepSeek validation SHA256 values
matched = deepseek_preds[["SHA256", "Actual"]].merge(
    df[["SHA256", "Type"]],
    on="SHA256",
    how="left",
    indicator=True
)

print("DeepSeek validation samples:", len(deepseek_preds))
print("Matched samples:", (matched["_merge"] == "both").sum())
print("Missing SHA256:", (matched["_merge"] == "left_only").sum())

# Check whether actual labels agree
matched = matched[matched["_merge"] == "both"].copy()
matched["Label_Match"] = matched["Actual"] == matched["Type"]

print("Label agreement:", matched["Label_Match"].sum())
print("Label mismatches:", (~matched["Label_Match"]).sum())

if (~matched["Label_Match"]).any():
    print("\nMismatch examples:")
    print(matched.loc[~matched["Label_Match"], ["SHA256", "Actual", "Type"]].head(10))

In [ ]:
LABEL_MAP = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

matched["Type_Name"] = matched["Type"].map(LABEL_MAP)

print("Label agreement:",
      (matched["Actual"] == matched["Type_Name"]).sum())

print("Label mismatches:",
      (matched["Actual"] != matched["Type_Name"]).sum())

print("\nActual label distribution:")
print(matched["Actual"].value_counts().sort_index())

print("\nCurrent dataset label distribution:")
print(matched["Type_Name"].value_counts().sort_index())

In [ ]:
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Reconstruct DeepSeek's exact validation set
val_df_exact = deepseek_preds[["SHA256", "Actual"]].merge(
    df,
    on="SHA256",
    how="left"
)

# Verify ordering and labels
assert len(val_df_exact) == 5898
assert (val_df_exact["Actual"] == val_df_exact["Type"].map(LABEL_MAP)).all()

# Reconstruct training data using the original 60:20:20 split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    stratify=df["Type"],
    random_state=42
)

# Exclude validation samples from training
val_hashes = set(val_df_exact["SHA256"])
train_df_exact = train_df[~train_df["SHA256"].isin(val_hashes)].copy()

# Prepare features
feature_cols = [c for c in df.columns if c not in ["SHA256", "Type"]]

X_train = train_df_exact[feature_cols]
y_train = train_df_exact["Type"]

X_val = val_df_exact[feature_cols]
y_val = val_df_exact["Type"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Feature count:", len(feature_cols))

In [ ]:
original_train_hashes = set(train_df["SHA256"])
deepseek_val_hashes = set(val_df_exact["SHA256"])

overlap = original_train_hashes.intersection(deepseek_val_hashes)

print("Original reconstructed training samples:", len(train_df))
print("DeepSeek validation samples:", len(val_df_exact))
print("Overlapping samples:", len(overlap))
print("Non-overlapping training samples:", len(train_df) - len(overlap))

# Check whether the original reconstructed validation split matches
original_val_df = temp_df.iloc[0:0]  # placeholder

print("\nDeepSeek validation samples found in reconstructed training:",
      len(overlap))

In [ ]:
import pandas as pd

partial_path = (
    "/kaggle/working/deepseek_validation_results/"
    "validation_predictions_partial.csv"
)

partial = pd.read_csv(partial_path)

print("Shape:", partial.shape)
print("\nColumns:", partial.columns.tolist())
print("\nFirst 10 rows:")
display(partial.head(10))

if "SHA256" in partial.columns:
    print("\nUnique SHA256:", partial["SHA256"].nunique())
    print(
        "Overlap with complete DeepSeek validation:",
        partial["SHA256"].isin(deepseek_preds["SHA256"]).sum()
    )

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    level = root.replace("/kaggle/working", "").count(os.sep)

    if level > 3:
        dirs[:] = []
        continue

    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os

path = "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"

print("Exists:", os.path.exists(path))
print("File size:", os.path.getsize(path), "bytes")

with open(path, "rb") as f:
    content = f.read(1000)

print("\nFirst 1000 bytes:")
print(repr(content))

In [ ]:
path = "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"

with open(path, "r", encoding="utf-8") as f:
    source = f.read()

keywords = [
    "train_test_split",
    "random_state",
    "stratify",
    "train_df",
    "val_df",
    "test_df",
    "validation_df",
    "DATASET_PATH",
    "read_csv",
    "SHA256"
]

lines = source.splitlines()

for keyword in keywords:
    matches = [
        i for i, line in enumerate(lines)
        if keyword.lower() in line.lower()
    ]

    if matches:
        print(f"\n{'='*65}")
        print(f"KEYWORD: {keyword}")
        print('='*65)

        for i in matches:
            start = max(0, i - 5)
            end = min(len(lines), i + 9)

            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")
            print()

In [ ]:
from sklearn.model_selection import train_test_split

# Stage 1: 80% development, 20% test
development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Type"]
)

# Stage 2: 75% training, 25% validation
train_df, validation_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=42,
    stratify=development_df["Type"]
)

# Reset indices
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Verify against DeepSeek's saved validation predictions
deepseek_hashes = set(deepseek_preds["SHA256"])

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

print("\nDeepSeek validation SHA256 matches:",
      validation_df["SHA256"].isin(deepseek_hashes).sum())

print("Training-validation overlap:",
      len(set(train_df["SHA256"]) & deepseek_hashes))

print("Test-validation overlap:",
      len(set(test_df["SHA256"]) & deepseek_hashes))

print("Exact validation match:",
      set(validation_df["SHA256"]) == deepseek_hashes)

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd
import numpy as np
import time

# Feature columns
feature_cols = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

# Prepare datasets
X_train = train_df[feature_cols]
y_train = train_df["Type"]

X_val = validation_df[feature_cols]
y_val = validation_df["Type"]

# Train Extra Trees
et_model = ExtraTreesClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

start = time.time()
et_model.fit(X_train, y_train)
training_time = time.time() - start

# Validation predictions
et_predictions = et_model.predict(X_val)

# Metrics
print("EXTRA TREES — ORIGINAL DEEPSEEK SPLIT")
print("=" * 55)

print(f"Training time: {training_time:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_val, et_predictions):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, et_predictions):.4f}")
print(f"Macro Precision: {precision_score(y_val, et_predictions, average='macro'):.4f}")
print(f"Macro Recall: {recall_score(y_val, et_predictions, average='macro'):.4f}")
print(f"Macro F1: {f1_score(y_val, et_predictions, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_val, et_predictions, average='weighted'):.4f}")

print("\nClassification Report:")
print(classification_report(y_val, et_predictions, target_names=[
    LABEL_MAP[i] for i in range(7)
]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, et_predictions))

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

OUTPUT_DIR = "/kaggle/working/deepseek_validation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create aligned prediction files
et_results = pd.DataFrame({
    "SHA256": validation_df["SHA256"].values,
    "Actual": validation_df["Type"].map(LABEL_MAP).values,
    "ExtraTrees_Predicted": pd.Series(et_predictions).map(LABEL_MAP).values
})

deepseek_results = deepseek_preds[
    ["SHA256", "Actual", "Predicted"]
].copy()

# Merge using SHA256 to guarantee sample alignment
comparison = deepseek_results.merge(
    et_results,
    on="SHA256",
    suffixes=("_DeepSeek", "_ET"),
    validate="one_to_one"
)

assert len(comparison) == 5898
assert (comparison["Actual_DeepSeek"] == comparison["Actual_ET"]).all()

comparison["DeepSeek_Correct"] = (
    comparison["Actual_DeepSeek"] == comparison["Predicted"]
)

comparison["ExtraTrees_Correct"] = (
    comparison["Actual_ET"] == comparison["ExtraTrees_Predicted"]
)

print("VALIDATION COMPARISON")
print("=" * 45)

print("DeepSeek correct:",
      comparison["DeepSeek_Correct"].sum())

print("Extra Trees correct:",
      comparison["ExtraTrees_Correct"].sum())

print("Both correct:",
      (comparison["DeepSeek_Correct"] &
       comparison["ExtraTrees_Correct"]).sum())

print("Only DeepSeek correct:",
      (comparison["DeepSeek_Correct"] &
       ~comparison["ExtraTrees_Correct"]).sum())

print("Only Extra Trees correct:",
      (~comparison["DeepSeek_Correct"] &
       comparison["ExtraTrees_Correct"]).sum())

print("Both incorrect:",
      (~comparison["DeepSeek_Correct"] &
       ~comparison["ExtraTrees_Correct"]).sum())

# Save results
comparison.to_csv(
    f"{OUTPUT_DIR}/model_comparison_predictions.csv",
    index=False
)

print("\nSaved: model_comparison_predictions.csv")

In [ ]:
from scipy.stats import binomtest

deepseek_only = int(
    (comparison["DeepSeek_Correct"] &
     ~comparison["ExtraTrees_Correct"]).sum()
)

et_only = int(
    (~comparison["DeepSeek_Correct"] &
     comparison["ExtraTrees_Correct"]).sum()
)

discordant = deepseek_only + et_only

result = binomtest(
    et_only,
    n=discordant,
    p=0.5,
    alternative="two-sided"
)

print("McNemar Exact Test")
print("=" * 45)
print("DeepSeek-only correct:", deepseek_only)
print("ExtraTrees-only correct:", et_only)
print("Discordant predictions:", discordant)
print("P-value:", result.pvalue)

if result.pvalue < 0.05:
    print("\nConclusion: The performance difference is statistically significant.")
else:
    print("\nConclusion: No statistically significant difference detected.")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 3:
        print("\n" + root)
        for file in files:
            print("  ", file)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Recreate the exact original split
development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Type"]
)

train_df, validation_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=42,
    stratify=development_df["Type"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nDataset split:")
print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

print("\nValidation class distribution:")
print(validation_df["Type"].value_counts().sort_index())

# Verify no overlap
assert set(train_df["SHA256"]).isdisjoint(validation_df["SHA256"])
assert set(train_df["SHA256"]).isdisjoint(test_df["SHA256"])
assert set(validation_df["SHA256"]).isdisjoint(test_df["SHA256"])

print("\nAll split integrity checks passed.")

In [ ]:
import torch
import transformers
import peft
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

In [ ]:
!pip install -q --no-cache-dir -U bitsandbytes

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_MODEL = "deepseek-ai/deepseek-coder-1.3b-base"

ADAPTER_PATH = (
    "/kaggle/input/datasets/mohdtahasayed2904/validation"
)

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Loading trained LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("\nDeepSeek model restored successfully!")
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

Loading tokenizer...


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [ ]:
!pip install -q sentencepiece

In [1]:
from transformers import AutoTokenizer

BASE_MODEL = "deepseek-ai/deepseek-coder-1.3b-base"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Vocabulary size:", len(tokenizer))

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

Tokenizer loaded: TokenizersBackend
Vocabulary size: 32022


In [3]:
from transformers import AutoTokenizer

import os

ADAPTER_PATH = "/kaggle/input/datasets/mohdtahasayed2904/validation"

adapter_tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH,
    trust_remote_code=True
)

print("Base tokenizer:", tokenizer.__class__.__name__)
print("Adapter tokenizer:", adapter_tokenizer.__class__.__name__)

print("Base vocabulary:", len(tokenizer))
print("Adapter vocabulary:", len(adapter_tokenizer))

test_text = "Predicted Malware Family: RedLineStealer"

base_ids = tokenizer.encode(test_text)
adapter_ids = adapter_tokenizer.encode(test_text)

print("\nToken IDs identical:", base_ids == adapter_ids)
print("Base IDs:", base_ids)
print("Adapter IDs:", adapter_ids)

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [4]:
import os
import json

ADAPTER_PATH = "/kaggle/input/datasets/mohdtahasayed2904/validation"

print("Adapter files:")
for f in os.listdir(ADAPTER_PATH):
    print(f)

# Inspect tokenizer configuration
config_path = os.path.join(ADAPTER_PATH, "tokenizer_config (4).json")

with open(config_path, "r") as f:
    config = json.load(f)

print("\nTokenizer configuration:")
print(json.dumps(config, indent=2)[:4000])

# Inspect tokenizer JSON
tokenizer_path = os.path.join(ADAPTER_PATH, "tokenizer (3).json")

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

print("\nTokenizer model type:",
      tokenizer_data.get("model", {}).get("type"))

print("Tokenizer vocabulary size:",
      len(tokenizer_data.get("model", {}).get("vocab", [])))

Adapter files:
adapter_config (3).json
adapter_model (6).safetensors
tokenizer (3).json
tokenizer_config (4).json

Tokenizer configuration:
{
  "backend": "tokenizers",
  "bos_token": "<\uff5cbegin\u2581of\u2581sentence\uff5c>",
  "clean_up_tokenization_spaces": false,
  "eos_token": "<\uff5cend\u2581of\u2581sentence\uff5c>",
  "is_local": false,
  "legacy": true,
  "local_files_only": false,
  "model_max_length": 16384,
  "pad_token": "<\uff5cend\u2581of\u2581sentence\uff5c>",
  "sp_model_kwargs": {},
  "tokenizer_class": "TokenizersBackend",
  "unk_token": null
}

Tokenizer model type: BPE
Tokenizer vocabulary size: 32000


In [5]:
import os
import shutil
from transformers import AutoTokenizer

ADAPTER_PATH = "/kaggle/input/datasets/mohdtahasayed2904/validation"
TOKENIZER_PATH = "/kaggle/working/adapter_tokenizer"

os.makedirs(TOKENIZER_PATH, exist_ok=True)

shutil.copy(
    os.path.join(ADAPTER_PATH, "tokenizer (3).json"),
    os.path.join(TOKENIZER_PATH, "tokenizer.json")
)

shutil.copy(
    os.path.join(ADAPTER_PATH, "tokenizer_config (4).json"),
    os.path.join(TOKENIZER_PATH, "tokenizer_config.json")
)

adapter_tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH,
    trust_remote_code=True
)

print("Adapter tokenizer loaded successfully!")
print("Tokenizer class:", type(adapter_tokenizer).__name__)
print("Vocabulary size:", len(adapter_tokenizer))
print("EOS token:", adapter_tokenizer.eos_token)
print("BOS token:", adapter_tokenizer.bos_token)
print("PAD token:", adapter_tokenizer.pad_token)

Adapter tokenizer loaded successfully!
Tokenizer class: TokenizersBackend
Vocabulary size: 32022
EOS token: <｜end▁of▁sentence｜>
BOS token: <｜begin▁of▁sentence｜>
PAD token: <｜end▁of▁sentence｜>


In [6]:
# Compare complete vocabulary mappings
base_vocab = tokenizer.get_vocab()
adapter_vocab = adapter_tokenizer.get_vocab()

print("Base vocabulary entries:", len(base_vocab))
print("Adapter vocabulary entries:", len(adapter_vocab))

print("Identical vocabulary mappings:", base_vocab == adapter_vocab)

# Compare tokenization on representative inputs
test_texts = [
    "You are a Windows PE malware classification assistant.",
    "Predicted Malware Family: SnakeKeyLogger",
    "CreateFileA=1, RegOpenKeyExA=0, VirtualAlloc=1",
    "<API_FUNCTIONS> CreateFileA=1 </API_FUNCTIONS>"
]

print("\nTokenization comparison:")

for text in test_texts:
    base_ids = tokenizer.encode(text, add_special_tokens=False)
    adapter_ids = adapter_tokenizer.encode(text, add_special_tokens=False)

    print("\nText:", text)
    print("Identical token IDs:", base_ids == adapter_ids)

Base vocabulary entries: 32022
Adapter vocabulary entries: 32022
Identical vocabulary mappings: True

Tokenization comparison:

Text: You are a Windows PE malware classification assistant.
Identical token IDs: True

Text: Predicted Malware Family: SnakeKeyLogger
Identical token IDs: True

Text: CreateFileA=1, RegOpenKeyExA=0, VirtualAlloc=1
Identical token IDs: True

Text: <API_FUNCTIONS> CreateFileA=1 </API_FUNCTIONS>
Identical token IDs: True


In [7]:
import json
import os

ADAPTER_PATH = "/kaggle/input/datasets/mohdtahasayed2904/validation"

config_path = os.path.join(ADAPTER_PATH, "adapter_config (3).json")

with open(config_path, "r") as f:
    adapter_config = json.load(f)

print(json.dumps(adapter_config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "deepseek-ai/deepseek-coder-1.3b-base",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.21.2",
  "qalora_group_size": 16,
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "q_proj",
    "up_proj",
    "gate_proj",
    "v_proj",
    "k_proj",
    "down_proj",
    "o_proj"
  ],
  "target_para

In [8]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "deepseek-ai/deepseek-coder-1.3b-base"

ADAPTER_PATH = "/kaggle/input/datasets/mohdtahasayed2904/validation"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Base model loaded.")

print("\nLoading trained LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)

model.eval()

print("\nDeepSeek malware classifier restored successfully!")
print("Device:", model.device)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

Loading base model...


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.69GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.69GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Base model loaded.

Loading trained LoRA adapter...


ValueError: Can't find 'adapter_config.json' at '/kaggle/input/datasets/mohdtahasayed2904/validation'

In [9]:
import os
import shutil

SOURCE_PATH = "/kaggle/input/datasets/mohdtahasayed2904/validation"
RESTORED_PATH = "/kaggle/working/restored_deepseek_adapter"

os.makedirs(RESTORED_PATH, exist_ok=True)

file_mapping = {
    "adapter_config (3).json": "adapter_config.json",
    "adapter_model (6).safetensors": "adapter_model.safetensors",
    "tokenizer (3).json": "tokenizer.json",
    "tokenizer_config (4).json": "tokenizer_config.json"
}

for source_file, destination_file in file_mapping.items():
    shutil.copy2(
        os.path.join(SOURCE_PATH, source_file),
        os.path.join(RESTORED_PATH, destination_file)
    )

print("Restored adapter directory:")
for filename in os.listdir(RESTORED_PATH):
    print(filename)

Restored adapter directory:
adapter_config.json
tokenizer_config.json
tokenizer.json
adapter_model.safetensors


In [10]:
model = PeftModel.from_pretrained(
    base_model,
    RESTORED_PATH,
    is_trainable=False
)

model.eval()

print("DeepSeek adapter loaded successfully!")
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

DeepSeek adapter loaded successfully!
Trainable parameters: 0


In [12]:
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("\nTarget distribution:")
print(df["Type"].value_counts().sort_index())

Dataset loaded successfully!
Shape: (29489, 320)
Columns: 320

Target distribution:
Type
0    1877
1    5022
2    4643
3    4955
4    5076
5    4217
6    3699
Name: count, dtype: int64


In [13]:
feature_columns = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

print("Total features:", len(feature_columns))

groups = {
    "API Functions": feature_columns[:200],
    "DLL Imports": feature_columns[200:227],
    "PE Header": feature_columns[227:273],
    "PE Sections": feature_columns[273:318]
}

for name, columns in groups.items():
    print(f"\n{name}: {len(columns)} features")
    print("First 5:", columns[:5])
    print("Last 5:", columns[-5:])

assert sum(len(v) for v in groups.values()) == 318

Total features: 318

API Functions: 200 features
First 5: ['virtualallocex', 'setunhandledexceptionfilter', 'getprocaddress', 'getcurrentprocessid', 'setlasterror']
Last 5: ['vbavartstne', 'vbalatememcall', 'buildcommdcbandtimeoutsa', 'getconsolealiaseslengtha', 'vbastrvarval']

DLL Imports: 27 features
First 5: ['advapi32.dll', 'kernel32.dll', 'ole32.dll', 'oleaut32.dll', 'shlwapi.dll']
Last 5: ['oleacc.dll', 'msvcp90.dll', 'msvbvm60.dll', 'mfc42.dll', 'shfolder.dll']

PE Header: 46 features
First 5: ['e_cblp', 'e_cp', 'e_crlc', 'e_cparhdr', 'e_minalloc']
Last 5: ['SizeOfHeaders', 'CheckSum', 'Subsystem', 'DllCharacteristics', 'SizeOfStackReserve']

PE Sections: 45 features
First 5: ['text_Misc_VirtualSize', 'text_VirtualAddress', 'text_SizeOfRawData', 'text_PointerToRawData', 'text_Characteristics']
Last 5: ['pdata_Misc_VirtualSize', 'pdata_VirtualAddress', 'pdata_SizeOfRawData', 'pdata_PointerToRawData', 'pdata_Characteristics']


In [14]:
import os
import re
import torch
import pandas as pd
import numpy as np

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

LABEL_MAP = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

ID_TO_LABEL = LABEL_MAP

BATCH_SIZE = 2
MAX_NEW_TOKENS = 20

OUTPUT_DIR = "/kaggle/working/deepseek_validation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 2. VERIFY DATA AND MODEL
# ============================================================

assert "df" in globals(), "Dataset df is missing."
assert "validation_df" in globals(), "validation_df is missing."
assert "model" in globals(), "DeepSeek model is missing."
assert "adapter_tokenizer" in globals(), "Adapter tokenizer is missing."

assert len(validation_df) == 5898
assert len(df.columns) == 320

print("Dataset:", df.shape)
print("Validation samples:", len(validation_df))
print("Model loaded:", model is not None)

# ============================================================
# 3. FEATURE GROUPS
# ============================================================

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

assert len(FEATURE_COLUMNS) == 318

API_FEATURES = FEATURE_COLUMNS[:200]
DLL_FEATURES = FEATURE_COLUMNS[200:227]
HEADER_FEATURES = FEATURE_COLUMNS[227:273]
SECTION_FEATURES = FEATURE_COLUMNS[273:318]

print("\nFeature groups verified:")
print("API Functions:", len(API_FEATURES))
print("DLL Imports:", len(DLL_FEATURES))
print("PE Header:", len(HEADER_FEATURES))
print("PE Sections:", len(SECTION_FEATURES))

# ============================================================
# 4. ORIGINAL CLASSIFICATION PROMPT
# ============================================================

SYSTEM_PROMPT = """You are a Windows PE malware classification assistant.

Classify the executable using only its provided static PE features.

Instructions:
1. Analyze all provided feature values.
2. Zero values are meaningful and must not be ignored.
3. Do not invent missing information.
4. Do not infer unsupported runtime behavior.
5. Select exactly one malware family.

Available classes:
Benign, RedLineStealer, Downloader, RAT,
BankingTrojan, SnakeKeyLogger, Spyware."""

def format_features(row, columns):
    return "\n".join(
        f"{col}={row[col]}" for col in columns
    )

def create_prompt(row):

    return f"""### SYSTEM
{SYSTEM_PROMPT}

### USER
Classify the following Windows PE executable.

<API_FUNCTIONS>
{format_features(row, API_FEATURES)}
</API_FUNCTIONS>

<DLL_IMPORTS>
{format_features(row, DLL_FEATURES)}
</DLL_IMPORTS>

<PE_HEADER>
{format_features(row, HEADER_FEATURES)}
</PE_HEADER>

<PE_SECTIONS>
{format_features(row, SECTION_FEATURES)}
</PE_SECTIONS>

### ASSISTANT:"""

# ============================================================
# 5. TOKENIZER AND MODEL SETTINGS
# ============================================================

adapter_tokenizer.padding_side = "left"
adapter_tokenizer.pad_token = adapter_tokenizer.eos_token

model.eval()

print("\nModel ready for inference.")
print("Device:", model.device)

# ============================================================
# 6. PREDICTION EXTRACTION
# ============================================================

# Longest labels first to avoid ambiguous substring matches.
LABELS_SORTED = sorted(
    LABEL_MAP.items(),
    key=lambda x: len(x[1]),
    reverse=True
)

def extract_prediction(text):

    for label_id, label_name in LABELS_SORTED:

        pattern = r"(?<![A-Za-z])" + re.escape(label_name) + r"(?![A-Za-z])"

        if re.search(pattern, text, flags=re.IGNORECASE):
            return label_id

    return -1

# ============================================================
# 7. GENERATE VALIDATION PREDICTIONS
# ============================================================

results = []

validation_records = validation_df.to_dict("records")

print("\nStarting validation inference...")

for start in tqdm(
    range(0, len(validation_records), BATCH_SIZE)
):

    batch_records = validation_records[
        start:start + BATCH_SIZE
    ]

    prompts = [
        create_prompt(row)
        for row in batch_records
    ]

    inputs = adapter_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=adapter_tokenizer.pad_token_id,
            eos_token_id=adapter_tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        :, inputs["input_ids"].shape[1]:
    ]

    decoded_outputs = adapter_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    for row, generated_text in zip(
        batch_records,
        decoded_outputs
    ):

        predicted_label = extract_prediction(
            generated_text
        )

        results.append({
            "SHA256": row["SHA256"],
            "Actual": int(row["Type"]),
            "Predicted": predicted_label,
            "Actual_Name": LABEL_MAP[int(row["Type"])],
            "Predicted_Name": LABEL_MAP.get(
                predicted_label, "Unclassified"
            ),
            "Generated_Text": generated_text.strip()
        })

# ============================================================
# 8. SAVE PREDICTIONS
# ============================================================

predictions_df = pd.DataFrame(results)

prediction_path = os.path.join(
    OUTPUT_DIR,
    "deepseek_validation_predictions.csv"
)

predictions_df.to_csv(
    prediction_path,
    index=False
)

print("\nInference completed.")

print("Total predictions:", len(predictions_df))

print(
    "Unclassified predictions:",
    (predictions_df["Predicted"] == -1).sum()
)

print("Saved:", prediction_path)

# ============================================================
# 9. EVALUATION
# ============================================================

valid_predictions = predictions_df[
    predictions_df["Predicted"] != -1
]

y_true = valid_predictions["Actual"]
y_pred = valid_predictions["Predicted"]

print("\n" + "=" * 60)
print("DEEPSEEK VALIDATION RESULTS")
print("=" * 60)

print("Accuracy:",
      round(accuracy_score(y_true, y_pred), 4))

print("Balanced Accuracy:",
      round(balanced_accuracy_score(y_true, y_pred), 4))

print("Macro Precision:",
      round(precision_score(
          y_true, y_pred, average="macro", zero_division=0
      ), 4))

print("Macro Recall:",
      round(recall_score(
          y_true, y_pred, average="macro", zero_division=0
      ), 4))

print("Macro F1:",
      round(f1_score(
          y_true, y_pred, average="macro", zero_division=0
      ), 4))

print("Weighted F1:",
      round(f1_score(
          y_true, y_pred, average="weighted", zero_division=0
      ), 4))

print("\nClassification Report:\n")

print(classification_report(
    y_true,
    y_pred,
    labels=list(LABEL_MAP.keys()),
    target_names=list(LABEL_MAP.values()),
    zero_division=0
))

# ============================================================
# 10. SAVE METRICS AND CONFUSION MATRIX
# ============================================================

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "macro_precision": precision_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "weighted_f1": f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    ),
    "total_samples": len(predictions_df),
    "unclassified": int(
        (predictions_df["Predicted"] == -1).sum()
    )
}

import json

with open(
    os.path.join(OUTPUT_DIR, "deepseek_validation_metrics.json"),
    "w"
) as f:
    json.dump(metrics, f, indent=4)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(LABEL_MAP.keys())
)

cm_df = pd.DataFrame(
    cm,
    index=list(LABEL_MAP.values()),
    columns=list(LABEL_MAP.values())
)

cm_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "deepseek_validation_confusion_matrix.csv"
    )
)

print("\nAll validation artifacts saved successfully.")

AssertionError: validation_df is missing.